# **Tarea 1 – Analítica de Datos (MACI 2026)**

**Alumno:** Keyden Urbina

**Mercado de celulares en Chile (solotodo.cl) y modelo de rango de precios**

## Notebook 01 · Web scraping → `data/output/data_raw.csv`

Extrae **solo desde el HTML** de solotodo.cl (requests + BeautifulSoup, **sin la API pública**) las características, características avanzadas, accesorios y los precios de cada celular en todas las tiendas, y si es reacondicionado o no. Guarda los datos **sin limpiar**.

| Indicación del curso | Cómo se cumple |
|---|---|
| 0. Guardar lo descargado a medida que avanzas | Cada página se guarda comprimida en `data/input/html_AAAAMMDD/` apenas se descarga. Si el proceso se corta, al re-ejecutar se lee del disco y solo se baja lo que falta. |
| 1. Paginación `?page=N` y `time.sleep(1)` | El listado se recorre con `/celulares?page=1, 2, 3…` hasta que una página no trae celulares nuevos. Hay 1 s de pausa entre requests. |
| 2. Páginas de modelo (`/iphone-17`) | Se usa la variante que la página **muestra seleccionada al hacer clic** en la tarjeta (el `?product=…` del enlace), como pide el enunciado. Una tabla compara esa variante con la primera de la lista del `<script>`. |
| 3. Precios ocultos ("Ver más precios") | Las ofertas de **todas** las tiendas salen del `<script type="application/ld+json">` (tienda y condición). El precio normal, el precio oferta, el cupón y si la oferta exige plan salen de los datos de cada oferta que la página trae embebidos en sus `<script>`. Ambos están en el HTML. |
| 4. Números desde texto | Se resuelve en el notebook 02 (limpieza), con expresiones regulares. |
| 5. ≥ 100 productos de distintas páginas | Se recorre el catálogo completo: todas las páginas del listado. |
| 6. requests + BeautifulSoup, sin API | Toda la información sale del HTML de las páginas públicas del sitio. |

### Paso 0: Importar las librerías y rutas necesarias

In [1]:
# Librerías a utilizar:
import re, json, time, gzip
import pandas as pd, requests
from bs4 import BeautifulSoup                         # lee HTML: esqueleto de la página web
from requests.adapters import HTTPAdapter, Retry      # reintentos automáticos si el sitio falla
from tqdm.auto import tqdm                            # barra de progreso
from pathlib import Path

In [2]:
# Carpeta "data" del proyecto:
DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), Path("data"))
OUT = DATA / "output"                                 # aquí queda data_raw.csv
ahora = pd.Timestamp.now(tz="America/Santiago")       # hora de Chile (Colab corre en hora UTC)
CACHE = DATA / "input" / f"html_{ahora:%Y%m%d}"       # HTML descargado hoy: si el proceso se corta, se retoma desde aquí
for carpeta in (OUT, CACHE): carpeta.mkdir(parents=True, exist_ok=True)
marca = CACHE / "inicio.txt"                          # anota la hora en que empezó la descarga de hoy (solo la 1ª vez)
if not marca.exists(): marca.write_text(ahora.isoformat())
inicio = pd.Timestamp(marca.read_text())              # al re-ejecutar se usa la hora de la descarga original, no la de ahora
print("RAW →", OUT.resolve(), "\nHTML →", CACHE.resolve(), "\nDescarga iniciada:", f"{inicio:%Y-%m-%d %H:%M}")

RAW → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/output 
HTML → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/input/html_20261108 
Descarga iniciada: 2026-11-08 11:55


### Paso 1: Descarga con respaldo en disco
`bajar()` es la función para la descarga de la página. La guarda comprimida con 1 seg de espera para no saturar o bloquear la extracción. Si esa página ya se bajó hoy, la lee del disco y no vuelve a pedirla.

In [3]:
BASE = "https://www.solotodo.cl"
s = requests.Session()                                # reutiliza la conexión: más rápido que abrir una por página
s.headers["User-Agent"] = "Mozilla/5.0"               # nos presentamos como un navegador común
s.mount("https://", HTTPAdapter(max_retries=Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))

def bajar(ruta, nombre):
    """HTML de BASE + ruta. Si ya se bajó hoy, lo lee del disco; si no, lo descarga, lo guarda comprimido y espera 1 s."""
    archivo = CACHE / f"{nombre}.html.gz"
    if archivo.exists():                              # ya descargado → no se vuelve a pedir al sitio
        return gzip.open(archivo, "rt", encoding="utf-8").read()
    r = s.get(BASE + ruta, timeout=30)
    r.raise_for_status()                              # si el sitio responde error (403, 404…), se detiene en vez de guardar basura
    with gzip.open(archivo, "wt", encoding="utf-8") as f: f.write(r.text)   # se guarda apenas se descarga
    time.sleep(1)                                     # pausa de 1 s entre requests, para no sobrecargar el sitio
    return r.text

### Paso 2: Listado de celulares (paginación)

In [4]:
# Listado: recorre /celulares?page=1, 2, 3… y junta los enlaces de cada tarjeta (20 por página)
enlaces, pag = [], 1
while True:
    sopa = BeautifulSoup(bajar(f"/celulares?page={pag}", f"listado_{pag:02d}"), "html.parser")
    hrefs = [a["href"] for a in sopa.find_all("a", href=True)]
    nuevos = [h for h in dict.fromkeys(hrefs)                                  # dict.fromkeys: quita repetidos y mantiene el orden
              if (h.startswith("/products/") or "?product=" in h) and h not in enlaces]   # tarjetas: producto o página de modelo
    if not nuevos: break                                                       # página sin celulares nuevos → se acabó el listado
    enlaces += nuevos
    pag += 1
modelos = [h for h in enlaces if not h.startswith("/products/")]
print(f"{len(enlaces)} celulares en {pag - 1} páginas | {len(modelos)} enlaces a página de modelo (ej: {modelos[:2]})")

242 celulares en 13 páginas | 13 enlaces a página de modelo (ej: ['/iphone-16?product=254399', '/iphone-17?product=355682'])


### Paso 3: De cada enlace a su ficha, con todas sus ofertas
Se prueba primero con **1 producto**, de acuerdo a la recomendación de la tarea.

In [5]:
# Enlace → id del producto.
#  · Tarjeta normal:      "/products/264814-samsung-…"   → 264814
#  · Página de modelo:    "/iphone-17?product=355682"     → 355682
#    El parámetro product es la variante que la página muestra seleccionada al hacer clic en la tarjeta (se comprueba en la tabla de más abajo)

def id_producto(href):
    return re.search(r"(?:/products/|product=)(\d+)", href)[1]

# Ficha técnica por sección: cada <h3> (Características, Características avanzadas, Accesorios) va seguido de un <dl>
# con pares <dt>etiqueta</dt><dd>valor</dd>. Se leen dentro de su sección, así un <dt> nunca se cruza con el <dd> de otra.

SECCION = {}                                                                   # etiqueta → sección (para el control de cobertura)
def ficha(sopa):
    datos = {}
    for h3 in sopa.find_all("h3"):
        dl = h3.find_next_sibling("dl")
        if dl is None: continue                                                # <h3> que no es de la ficha técnica
        for dt in dl.find_all("dt"):
            campo = dt.get_text(strip=True)
            SECCION[campo] = h3.get_text(strip=True)
            datos[campo] = " ".join(dt.find_next_sibling("dd").get_text().split())   # " ".join(...split()) deja el texto en una sola línea
    return datos

# Ficha del producto (todo sale del HTML):
#  a) características, avanzadas y accesorios → ficha()
#  b) ofertas de TODAS las tiendas (no solo las 5 visibles) → <script type="application/ld+json">: tienda, URL y condición
#  c) precio normal, precio oferta, cupón y tipo de oferta (liberado o con plan) → datos de cada oferta que la página
#     trae embebidos en sus <script> (self.__next_f.push). Se cruzan con (b) por la URL de la tienda.

def producto(pid):
    html = bajar(f"/products/{pid}", f"producto_{pid}")
    sopa = BeautifulSoup(html, "html.parser")
    tag = sopa.find("script", type="application/ld+json")
    ld = json.loads(tag.string) if tag else {}                                 # sin ofertas, la página no trae ld+json
    marca = ld.get("brand", {})
    base = {"nombre": ld.get("name") or sopa.h1.get_text(strip=True),
            "marca": marca.get("name") if isinstance(marca, dict) else marca,
            "id": pid, **ficha(sopa)}
    tienda = {o["url"]: o["seller"]["name"] for o in ld.get("offers", {}).get("offers", [])}   # URL de la oferta → nombre de la tienda

    # (c) Los <script> self.__next_f.push([1,"…"]) guardan texto escapado: json.loads lo desescapa y se unen en uno solo

    texto = "".join(json.loads(f'"{t}"') for t in re.findall(r'self\.__next_f\.push\(\[1,"((?:[^"\\]|\\.)*)"\]\)', html))
    i = texto.find('"entities":')
    ofertas = json.JSONDecoder().raw_decode(texto, i + len('"entities":'))[0] if i >= 0 else []   # lee solo la lista [...]

    filas = [{**base, "tienda": tienda.get(e["external_url"]),
              "id_oferta": e["id"],                                            # identificador de la publicación en solotodo (trazabilidad)
              "url_oferta": e["external_url"],                                 # enlace a la publicación en la tienda
              "precio_normal": e["active_registry"]["normal_price"],
              "precio_oferta": e["active_registry"]["offer_price"],
              "condicion": e["condition"],                                     # NewCondition / RefurbishedCondition / UsedCondition
              "cupon_tipo": (e["best_coupon"] or {}).get("amount_type_text"),  # cupón de descuento, si existe
              "cupon_monto": (e["best_coupon"] or {}).get("amount"),
              "tipo_oferta": e["cell_plan"]["name"] if e["cell_plan"] else "Liberado"}   # "Liberado" o el plan que exige (ej. "WOM Prepago")
             for e in ofertas]                                                 # TODAS las ofertas (RAW sin limpiar); las con plan se filtran en el 02
    return filas or [base]                                                     # sin ofertas: queda la ficha sin precios

# Prueba con 1 producto antes de recorrer todo (recomendación de la tarea)

prueba = pd.DataFrame(producto(id_producto(enlaces[0])))
prueba[["nombre", "marca", "tienda", "precio_normal", "precio_oferta", "condicion", "cupon_tipo", "tipo_oferta"]]

,nombre,marca,tienda,precio_normal,precio_oferta,condicion,cupon_tipo,tipo_oferta
0,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Movistar,549990.00,549990.00,https://schema.org/RefurbishedCondition,None,Liberado
1,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Reuse,739990.00,739990.00,https://schema.org/RefurbishedCondition,Percentage,Liberado
2,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Digitek,759990.00,759990.00,https://schema.org/UsedCondition,None,Liberado
3,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Lider,799990.00,799990.00,https://schema.org/NewCondition,None,Liberado
4,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Movistar,879990.00,879990.00,https://schema.org/NewCondition,None,Liberado
5,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Falabella,949990.00,899990.00,https://schema.org/NewCondition,None,Liberado
6,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Tienda Entel,1469990.00,999990.00,https://schema.org/NewCondition,None,Liberado
7,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,WOM,1084800.00,1084800.00,https://schema.org/NewCondition,None,WOM Prepago
8,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,ABC,1099990.00,1099990.00,https://schema.org/NewCondition,None,Liberado
9,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,Ripley,1179990.00,1159990.00,https://schema.org/NewCondition,None,Liberado


#### Comprobación: ¿qué variante muestra la página al hacer clic?

Se pide la variante que la página **muestra por defecto al hacer clic** en el celular. Para las páginas de modelo se descarga el enlace **tal como se abre al hacer clic** (con `?product=…`) y se compara:

- **Variante mostrada:** el nombre que trae la página en su `ld+json` y la memoria interna de su ficha.

- **Primera de la lista:** la primera de `"products":[…]` en el `<script>`. Es la que tomaba la versión anterior de este notebook.

La tabla queda guardada en `data/output/comprobacion_variantes.csv`.

In [6]:
# Para cada página de modelo: la página tal como se abre al hacer clic vs. la primera variante de la lista interna
filas_mod = []
for h in modelos:
    html = bajar(h, "modelo_" + h.split("?")[0].strip("/"))                    # el enlace completo, con ?product=…
    sopa = BeautifulSoup(html, "html.parser")
    tag = sopa.find("script", type="application/ld+json")                      # datos de la variante seleccionada
    primera = re.search(r'"products":\[\{"id":(\d+),"name":"([^"]+)"', html.replace('\\"', '"'))   # comillas escapadas: \" → "
    filas_mod.append({"enlace": h,
                      "id_seleccionado": id_producto(h),
                      "variante_mostrada": json.loads(tag.string)["name"] if tag else None,
                      "memoria_mostrada": ficha(sopa).get("Memoria interna"),
                      "primera_de_la_lista": primera[2],
                      "coincide": primera[1] == id_producto(h)})
comprobacion = pd.DataFrame(filas_mod)
comprobacion.to_csv(OUT / "comprobacion_variantes.csv", index=False)
print(f"La primera de la lista coincide con la variante mostrada en {comprobacion.coincide.sum()} de {len(comprobacion)} páginas de modelo")
comprobacion

La primera de la lista coincide con la variante mostrada en 4 de 13 páginas de modelo


,enlace,id_seleccionado,variante_mostrada,memoria_mostrada,primera_de_la_lista,coincide
0,/iphone-16?product=254399,254399,Apple iPhone 16 (128 GB / Black),128 GB,Apple iPhone 16 (128 GB / Black),True
1,/iphone-17?product=355682,355682,Apple iPhone 17 (256 GB / Black),256 GB,Apple iPhone 17 (256 GB / Black),True
2,/iphone-17-pro-max?product=355715,355715,Apple iPhone 17 Pro Max (256 GB / Cosmic Orange),256 GB,Apple iPhone 17 Pro Max (1 TB / Cosmic Orange),False
3,/samsung-galaxy-s25?product=264820,264820,Samsung Galaxy S25 (S931 / 256 GB / 12 GB / Na...,256 GB,Samsung Galaxy S25 (S931 / 128 GB / 12 GB / Bl...,False
4,/iphone-18-pro-max?product=406893,406893,Apple iPhone 18 Pro Max (256 GB / Burgundy) [M...,256 GB,Apple iPhone 18 Pro Max (1 TB / Black) [MJXX4B...,False
5,/iphone-15?product=221875,221875,Apple iPhone 15 (128 GB / Black),128 GB,Apple iPhone 15 (128 GB / Black),True
6,/iphone-17e?product=383360,383360,Apple iPhone 17e (256 GB / Black),256 GB,Apple iPhone 17e (256 GB / Black),True
7,/iphone-16e?product=267808,267808,Apple iPhone 16e (128 GB / White) [MD1R4BE/A],128 GB,Apple iPhone 16e (128 GB / Black) [MD1Q4BE/A],False
8,/iphone-18-pro?product=406922,406922,Apple iPhone 18 Pro (256 GB / Black) [MJRP4BE/A],256 GB,Apple iPhone 18 Pro (1 TB / Black) [MJRY4BE/A],False
9,/iphone-17-pro?product=355711,355711,Apple iPhone 17 Pro (256 GB / Deep Blue),256 GB,Apple iPhone 17 Pro (1 TB / Cosmic Orange) [MG...,False


**Interpretación:**

- La primera variante de la lista interna coincide con la que muestra la página en **solo 4 de 13** páginas de modelo (iPhone 16, 17, 17e y 15).

- En **7 de las 9 diferencias cambia la capacidad**:
  - En 6 iPhone de gama alta (17 Pro, 17 Pro Max, 18 Pro, 18 Pro Max, Air y Duo), la primera de la lista es la de **1 TB** y la página muestra la de **256 GB**.
  - En el Galaxy S25, la primera es la de 128 GB y la página muestra la de 256 GB.

- En las otras 2 (iPhone 16e y 14) solo cambia el color; la capacidad es la misma.

- **Conclusión:** usar la primera de la lista habría metido en el RAW la versión más cara de 6 iPhone y habría inflado el precio de Apple. Por eso se usa el `?product=` del enlace. La columna `memoria_mostrada` confirma que la página que se abre al hacer clic muestra esa variante.

#### Prueba con 2 páginas (~40 celulares)
Se procesan los celulares de las 2 primeras páginas del listado.

In [7]:
prueba2 = pd.DataFrame([f for h in enlaces[:40] for f in producto(id_producto(h))])   # ~2 páginas × 20 celulares
print(prueba2.shape, "| celulares:", prueba2.nombre.nunique(), "| tiendas:", prueba2.tienda.nunique(),
      "| ofertas con plan:", prueba2.tipo_oferta.notna().sum() - prueba2.tipo_oferta.eq("Liberado").sum(),
      "| celulares sin ofertas:", prueba2.precio_oferta.isna().sum(),
      "| ofertas sin nombre de tienda:", (prueba2.precio_oferta.notna() & prueba2.tienda.isna()).sum())   # debe ser 0

(392, 34) | celulares: 40 | tiendas: 38 | ofertas con plan: 28 | celulares sin ofertas: 0 | ofertas sin nombre de tienda: 0


### Paso 4: Todos los celulares → `data_raw.csv`

In [8]:
# Recorre todos los celulares y guarda data_raw.csv (1 fila = 1 celular en 1 tienda)
filas = []
for h in tqdm(enlaces):
    try:
        filas += producto(id_producto(h))
    except Exception as e:                                                     # un celular con problemas no detiene todo
        print("Error en", h, "→", e)

raw = pd.DataFrame(filas)
raw["fecha_extraccion"] = inicio.strftime("%Y-%m-%d %H:%M")                    # hora en que se descargaron los precios
archivo = f"data_raw_{inicio:%Y%m%d_%H%M}.csv"                                 # ej: data_raw_AAAAMMDD_HHMM.csv (re-ejecutar no crea otro)
raw.to_csv(OUT / archivo, index=False)                                         # copia histórica: cada ejecución crea un archivo nuevo
raw.to_csv(OUT / "data_raw.csv", index=False)                                  # versión vigente, con el nombre que pide el enunciado
print(raw.shape, "| celulares:", raw.nombre.nunique(), "→", OUT / archivo)
print("Ofertas por tipo:", raw.tipo_oferta.value_counts().to_dict())
raw.head()

  0%|          | 0/242 [00:00<?, ?it/s]

(1104, 35) | celulares: 242 → /Users/keydenurbina/Desktop/github/Proyecto Web Scraping/data/output/data_raw_20261108_1155.csv
Ofertas por tipo: {'Liberado': 1029, 'WOM Prepago': 36, 'Claro Prepago': 35, 'Movistar Prepago': 3}


,nombre,marca,id,Pantalla,Memoria interna,RAM,Cámara principal,Cámara frontal,Batería,Sistema operativo,...,tienda,id_oferta,url_oferta,precio_normal,precio_oferta,condicion,cupon_tipo,cupon_monto,tipo_oferta,fecha_extraccion
0,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Tienda Movistar,1956927.0,https://catalogo.movistar.cl/tienda/samsung-ga...,549990.00,549990.00,https://schema.org/RefurbishedCondition,None,None,Liberado,2026-11-08 11:55
1,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Reuse,2375370.0,https://www.reuse.cl/collections/celulares/pro...,739990.00,739990.00,https://schema.org/RefurbishedCondition,Percentage,5.00,Liberado,2026-11-08 11:55
2,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Digitek,2043624.0,https://digitek.cl/collections/semi-nuevos/pro...,759990.00,759990.00,https://schema.org/UsedCondition,None,None,Liberado,2026-11-08 11:55
3,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Lider,1866781.0,https://www.lider.cl/ip/telefonia/smartphone-g...,799990.00,799990.00,https://schema.org/NewCondition,None,None,Liberado,2026-11-08 11:55
4,Samsung Galaxy S25 Ultra (S938 / 256 GB / 12 G...,Samsung,264814,"Dynamic LTPO AMOLED 2X 6.9"" táctil (3120x1440)...",256 GB,12 GB,Estándar 200.0 MP + Teleobjetivo (Zoom) 10.0 M...,Estándar 12.0 MP,5000 mAh,Android 15,...,Tienda Movistar,1834799.0,https://catalogo.movistar.cl/tienda/samsung-ga...,879990.00,879990.00,https://schema.org/NewCondition,None,None,Liberado,2026-11-08 11:55


**Interpretación.**
- El RAW tiene **1.104 filas × 35 columnas**: 242 celulares de 13 páginas del listado, extraídos el **08-11-2026 a las 11:55**.

- **Tipo de oferta:** 1.029 liberadas y **74 con plan** (WOM Prepago 36, Claro Prepago 35 y Movistar Prepago 3). Se conservan sin filtrar y en el notebook 02 dejo solo las liberadas.

- **9 celulares se venden solo con plan** y 1 no tiene ninguna oferta (Vivo Y70 FE). Este último queda con su ficha y sin precio.

- **Condición:** 1.019 ofertas nuevas, 57 reacondicionadas y 27 usadas. **78 ofertas** tienen cupón.

### Paso 5: Controles de la extracción
1. **Variantes:** la variante mostrada al hacer clic en cada página de modelo está en el RAW.
2. **Cobertura de la ficha:** qué campos trae cada sección (características, avanzadas y accesorios), en qué % de los celulares y cuántos dicen "N/A".
3. **Contraste con el HTML:** en 3 celulares al azar, el número de pares `<dt>/<dd>` del HTML debe ser igual al de campos capturados.
4. **Trazabilidad:** cada oferta debe tener su propio `id_oferta`; si se repitiera, la extracción habría duplicado filas.


In [9]:
# Control 1 · la variante que se muestra al hacer clic es la que quedó en el RAW (mismo nombre)
print("Variantes de páginas de modelo presentes en el RAW:", comprobacion.variante_mostrada.isin(raw.nombre).sum(), "de", len(comprobacion))

# Control 2 · trazabilidad: cada oferta tiene su propio id_oferta (si se repitiera, la extracción habría duplicado filas)
print("Ofertas:", raw.id_oferta.notna().sum(), "| id_oferta distintos:", raw.id_oferta.nunique())

# Control 3 · contraste con el HTML: n° de <dt> en la página vs. campos capturados en el RAW
campos = [c for c in raw.columns if c in SECCION]                              # columnas que vienen de la ficha técnica
u = raw.drop_duplicates("id").set_index("id")                                  # 1 fila por celular
for pid in u.sample(3, random_state=1).index:
    sopa = BeautifulSoup(bajar(f"/products/{pid}", f"producto_{pid}"), "html.parser")   # se lee del disco, no se vuelve a pedir
    print(f"{u.nombre[pid][:45]:45} | <dt> en el HTML: {len(sopa.find_all('dt'))} | campos capturados: {u.loc[pid, campos].notna().sum()}")

# Control 4 · cobertura por sección: % de celulares que traen cada campo y cuántos dicen "N/A"
cobertura = pd.DataFrame({"sección": [SECCION[c] for c in campos],
                          "% de celulares con el campo": (u[campos].notna().mean() * 100).round(1).values,
                          'celulares con "N/A"': u[campos].eq("N/A").sum().values}, index=campos)
cobertura

Variantes de páginas de modelo presentes en el RAW: 13 de 13
Ofertas: 1103 | id_oferta distintos: 1103
Samsung Galaxy S24 Ultra (S928 / 256 GB / 12  | <dt> en el HTML: 22 | campos capturados: 22
Oppo Reno12 F (256 GB / 8 GB / Olive Green)   | <dt> en el HTML: 22 | campos capturados: 22
Honor Play 20A (128 GB / 4 GB / Midnight Blac | <dt> en el HTML: 22 | campos capturados: 22


,sección,% de celulares con el campo,"celulares con ""N/A"""
Pantalla,Características,100.0,0
Memoria interna,Características,100.0,0
RAM,Características,100.0,0
Cámara principal,Características,100.0,0
Cámara frontal,Características,100.0,4
Batería,Características,100.0,10
Sistema operativo,Características,100.0,0
Conectividad celular,Características,100.0,0
Procesador,Características avanzadas,100.0,6
CPU,Características avanzadas,100.0,6


**Interpretación:**

- **Variantes:** las 13 variantes que se muestran al hacer clic están en el RAW, con el mismo nombre.

- **Trazabilidad:** las 1.103 ofertas tienen 1.103 `id_oferta` distintos. La extracción no duplicó ninguna oferta: si dos filas se parecen, son publicaciones distintas (se analiza en el notebook 02).

- **Contraste con el HTML:** en los 3 celulares al azar, los 22 pares `<dt>/<dd>` de la página quedaron como 22 columnas. No se perdió ni se desalineó ningún campo.

- **Cobertura:** las 3 secciones vienen en los 242 celulares, con **22 campos**:
  - **Características (8):** pantalla, memoria interna, RAM, cámaras principal y frontal, batería, sistema operativo y conectividad celular.
  - **Características avanzadas (6):** procesador, CPU, GPU, peso, tamaño y soporte eSIM.
  - **Accesorios (8):** potencia de carga, almacenamiento externo, conector de audífonos, Wi-Fi, GPS, NFC, Bluetooth y lector de huellas.
  
- **Campos ausentes:** ninguno falta como etiqueta. Lo que falta son **valores**, que el sitio publica como "N/A": peso en 17 celulares, batería en 10, procesador, CPU y GPU en 6 cada uno, y cámara frontal en 4. Accesorios no tiene ningún "N/A". Los "N/A" se tratan en el notebook 02.

### Decisiones de la extracción:

- **Variante en las páginas de modelo:** se usa la que la página **muestra seleccionada al hacer clic** en la tarjeta (el `?product=…` del enlace). La tabla de comprobación muestra que la primera de la lista interna del `<script>` difiere en 9 de las 13 páginas de modelo; en 6 iPhone es la de 1 TB. Se verificó también en el navegador: al abrir el enlace queda seleccionada la variante del `?product=`, y la primera opción del selector "Capacidad" es la de menor capacidad.

- **Ofertas con plan de compañía:** se **conservan** en el RAW, con la columna `tipo_oferta` ("Liberado" o el nombre del plan, como lo muestra la ficha). Se excluyen en el notebook 02, porque su precio es el de un arriendo con plan (ej. \$9.990 + cuota mensual) y no el del equipo.

- **Celulares sin ofertas o solo con plan:** se conservan con su ficha. En la limpieza (notebook 02) se decide qué hacer con ellos.

- **Trazabilidad de las ofertas:** se guardan `id_oferta` y `url_oferta`, para distinguir una oferta repetida por la extracción de dos publicaciones distintas con el mismo precio.

- **Cupones:** se guardan tal cual (tipo y monto). El descuento se aplica en la limpieza, porque el enunciado considera el precio con cupón como precio oferta.

- **Duplicados y reacondicionados:** se guardan todos sin filtrar (el RAW va sin limpiar). Se tratan en el notebook 02.